# Pixels, filters, and image patches

**MedLIME Learning Track — Notebook 02/12** · Curriculum weeks 5–8 · Suggested time: 60–100 minutes

**Before starting:** Notebook 01 (arrays, masks).

**Learning goal:** Use NumPy indexing and patch masks to construct image perturbations.

> **Important:** All images below are synthetic visual teaching phantoms, **not real medical images**. The educational implementations are **not** the original MedLIME implementation, are not evaluated for clinical use, and do not establish diagnostic or explanation reliability.


In [ ]:
from pathlib import Path
import sys
here = Path.cwd()
for candidate in (here, here.parent, here.parent.parent):
    if (candidate / 'medlime_tutorials.py').exists():
        sys.path.insert(0, str(candidate.resolve())); break
else:
    raise FileNotFoundError('Open Jupyter from the MedLIME folder or its notebooks/ subfolder')
import numpy as np
import matplotlib.pyplot as plt
from medlime_tutorials import *
np.set_printoptions(precision=3, suppress=True)


## A picture is an array
Use array slices to edit square regions. In a 32×32 image, a 4×4 patch grid has 16 patches, each 8×8 pixels. LIME later treats each patch as an interpretable feature.

In [ ]:
image, lesion = synthetic_image(seed=14)
print('Top-left 4x4 pixel values:\n',image[:4,:4])
modified=image.copy(); modified[8:16,8:16]=0
fig=show_images([image,modified,np.abs(image-modified)],['original','edited (rows 8:16)','absolute change'])
plt.show()

## Build a binary patch mask
A one means retain the patch; zero means hide it. Unlike arbitrary pixel deletion, patches help maintain local structure and produce a small interpretable feature set.

In [ ]:
identifiers=patch_ids()
print('Distinct patch ids:',np.unique(identifiers))
keep=np.ones(16,np.float32); keep[5]=0; keep[10]=0
masked=patch_mask(image,keep,fill='zero')
fig=show_images([identifiers/15,image,masked],['patch numbers scaled 0..1','original','hide patches 5 and 10'],cmap='gray')
plt.show()

## Compare image processing choices
A Gaussian filter averages pixels nearby, reducing noise but also blurring small findings. A patch mask hides a selected region. Do not confuse *masking* with a true biological intervention.

In [ ]:
from scipy.ndimage import gaussian_filter
blurred=gaussian_filter(image, sigma=1.5)
fig=show_images([image,blurred,patch_mask(image,keep,fill='mean'),patch_mask(image,keep,fill='blur')],
 ['original','Gaussian smoothing','mean fill','blur fill'])
plt.show()
print('Mean intensity before/after smoothing:',round(float(image.mean()),3),round(float(blurred.mean()),3))

## What you should now understand
- An image pixel is numerical data and masks select subsets of pixels.
- A 4x4 patch grid gives 16 binary LIME features.
- Filters, missingness, and image perturbation are different operations.

## Try it yourself (do not just read the code)
1. Hide four chosen patches by setting their corresponding `keep` values to 0.
2. Change `sigma` and describe the risk to a very small lesion.

## Exit ticket
**Question:** Why might a black square be unlike anything a medical imaging model saw during training?

Record your answers in a new Markdown cell before moving to the next notebook. Many questions have more than one reasonable answer; explain your evidence.